# 决策树第六课：CART 决策树

本课目标：理解 CART 分类树如何用基尼系数选择划分。

一句话版：**CART 每次选择让划分后基尼指数最小的二叉划分。**

## 1. CART 和 ID3、C4.5 有什么不同？

前面学过：

- ID3：用信息增益选特征。
- C4.5：用信息增益率选特征。
- CART：常用基尼系数选划分。

CART 的全称是 Classification And Regression Tree，也就是分类与回归树。

它有两个重要特点：

- 既能做分类，也能做回归。
- 每次划分都是二叉划分，也就是一个节点只分成左右两个分支。

这节先学 CART 分类树。回归树后面单独讲。

## 2. 基尼系数 Gini 是什么？

Gini 系数表示的是：

$$从数据集D里随机抽取两个样本，它们属于不同类别的概率，它越小越好$$

分类树里，Gini 衡量一个节点的“不纯度”。

$$Gini(D)=1-\sum_{k=1}^{K}p_k^2$$

其中 $p_k$ 是第 $k$ 类样本在当前节点里的比例。

直觉：

- 如果一个节点全是同一类，Gini = 0，最纯。
- 如果二分类里两类各占一半，Gini = 0.5，最不纯。

例如一个节点里有 4 个批准、0 个拒绝：

$$Gini=1-(1^2+0^2)=0$$

如果有 2 个批准、2 个拒绝：

$$Gini=1-(0.5^2+0.5^2)=0.5$$

## 3. 划分后的 Gini 怎么算？

如果一个特征把数据分成左右两个分支，CART 会算加权后的 Gini：

$$GiniIndex(D,A)=\frac{|D_1|}{|D|}Gini(D_1)+\frac{|D_2|}{|D|}Gini(D_2)$$

注意它和条件熵很像：都是先算每个子节点的不纯度，再按样本数加权平均。

区别是：

- ID3/C4.5 常用熵。
- CART 分类树常用 Gini。

CART 选择的是：**加权 Gini 最小的划分。**

## 4. 继续使用贷款审批数据

| 编号 | 有工作 | 有房产 | 是否批准贷款 |
| ---: | --- | --- | --- |
| 1 | 是 | 是 | 批准 |
| 2 | 否 | 是 | 批准 |
| 3 | 是 | 否 | 批准 |
| 4 | 是 | 否 | 批准 |
| 5 | 否 | 否 | 拒绝 |
| 6 | 否 | 否 | 拒绝 |
| 7 | 是 | 是 | 批准 |
| 8 | 否 | 否 | 拒绝 |

我们比较两个二叉划分：

- 按 `有工作` 划分。
- 按 `有房产` 划分。

谁的加权 Gini 更小，CART 就先选谁。

## 5. 手算：按“有工作”划分

`有工作` 会把数据分成两组：

| 分支 | 样本数 | 批准 | 拒绝 |
| --- | ---: | ---: | ---: |
| 有工作 = 是 | 4 | 4 | 0 |
| 有工作 = 否 | 4 | 1 | 3 |

先算左分支 `有工作 = 是`：

$$Gini=1-(\frac{4}{4})^2-(\frac{0}{4})^2=0$$

再算右分支 `有工作 = 否`：

$$Gini=1-(\frac{1}{4})^2-(\frac{3}{4})^2$$

$$=1-\frac{1}{16}-\frac{9}{16}=\frac{6}{16}=0.375$$

最后按样本数量加权：

$$GiniIndex(D,有工作)=\frac{4}{8}\times0+\frac{4}{8}\times0.375=0.1875$$

## 6. 手算：按“有房产”划分

`有房产` 会把数据分成两组：

| 分支 | 样本数 | 批准 | 拒绝 |
| --- | ---: | ---: | ---: |
| 有房产 = 是 | 3 | 3 | 0 |
| 有房产 = 否 | 5 | 2 | 3 |

先算 `有房产 = 是`：

$$Gini=1-(\frac{3}{3})^2-(\frac{0}{3})^2=0$$

再算 `有房产 = 否`：

$$Gini=1-(\frac{2}{5})^2-(\frac{3}{5})^2$$

$$=1-\frac{4}{25}-\frac{9}{25}=\frac{12}{25}=0.48$$

最后按样本数量加权：

$$GiniIndex(D,有房产)=\frac{3}{8}\times0+\frac{5}{8}\times0.48=0.3$$

## 7. CART 选择哪个特征？

| 划分方式 | 加权 Gini |
| --- | ---: |
| 有工作 | 0.1875 |
| 有房产 | 0.3000 |

CART 选择加权 Gini 更小的划分，所以第一刀选择：

```text
有工作?
```

这和前面 ID3/C4.5 在这个例子上的选择一样，但原因不同：

- ID3/C4.5：看不确定性减少得多不多。
- CART：看划分后节点纯不纯。

## 8. 为什么选择加权 Gini 更小的划分？

Gini 衡量的是节点的不纯度。

```text
Gini = 0      节点最纯，全是一类
Gini 越大     节点越混，类别越杂
```

所以 CART 的目标不是让 Gini 变大，而是让划分后的两个子节点尽量纯。也就是说：**切完以后，左右两个分支整体越不混越好。**

这里要注意一个容易误会的地方：CART 不是选“某一个方向的 Gini 更小”，而是选“整次划分的加权 Gini 更小”。

比如按 `有工作` 划分：

```text
有工作 = 是：Gini = 0
有工作 = 否：Gini = 0.375
整次划分的加权 Gini = 0.1875
```

按 `有房产` 划分：

```text
有房产 = 是：Gini = 0
有房产 = 否：Gini = 0.48
整次划分的加权 Gini = 0.3000
```

虽然两个划分都有一个纯分支，Gini 都是 0，但 CART 比较的是整次划分后的加权结果：

```text
0.1875 < 0.3000
```

所以选择 `有工作`。

可以把 CART 的判断翻译成一句话：**我用哪个条件切一刀，能让切完后的左右两边整体最不像大杂烩？**

In [ ]:
from collections import Counter, defaultdict

loan_data = [
    {'有工作': '是', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '是否批准贷款': '批准'},
    {'有工作': '是', '有房产': '否', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
    {'有工作': '是', '有房产': '是', '是否批准贷款': '批准'},
    {'有工作': '否', '有房产': '否', '是否批准贷款': '拒绝'},
]

target = '是否批准贷款'
features = ['有工作', '有房产']

def gini(labels):
    total = len(labels)
    counts = Counter(labels)
    return 1 - sum((count / total) ** 2 for count in counts.values())

def split_by_feature(rows, feature):
    groups = defaultdict(list)
    for row in rows:
        groups[row[feature]].append(row)
    return groups

def gini_index(rows, feature, target):
    total = len(rows)
    groups = split_by_feature(rows, feature)
    return sum(
        (len(group_rows) / total) * gini([row[target] for row in group_rows])
        for group_rows in groups.values()
    )

for feature in features:
    print(f"特征：{feature}")
    for value, rows in split_by_feature(loan_data, feature).items():
        labels = [row[target] for row in rows]
        print(f"  {feature}={value}: {len(rows)}条, {dict(Counter(labels))}, Gini={gini(labels):.4f}")
    print(f"  加权 GiniIndex = {gini_index(loan_data, feature, target):.4f}\n")


## 9. CART 为什么强调二叉划分？

即使一个特征有多个取值，CART 也会把它拆成二叉问题。

比如 `天气` 有三个取值：晴、阴、雨。

CART 不会直接分成三叉：

```text
天气?
├── 晴
├── 阴
└── 雨
```

而是尝试类似这样的二叉划分：

```text
天气 是否属于 {晴} ?
├── 是
└── 否
```

或者：

```text
天气 是否属于 {晴, 阴} ?
├── 是
└── 否
```

然后选择加权 Gini 最小的那个二叉切法。

## 10. 多叉树和 CART 树的区别

多叉决策树允许一个节点一次分出多个分支。例如 `天气` 有三个取值时，多叉树可以直接这样分：

```text
天气?
├── 晴
├── 阴
└── 雨
```

CART 则固定使用二叉划分。它不会一次分出三条路，而是把问题变成二选一：

```text
天气 是否属于 {晴} ?
├── 是：晴
└── 否：阴、雨
```

如果右边还不纯，再继续二分：

```text
天气 是否属于 {阴} ?
├── 是：阴
└── 否：雨
```

所以两者的核心区别是：

| 类型 | 一次划分 | 典型算法 |
| --- | --- | --- |
| 多叉树 | 一个节点可以分出多个分支 | ID3、C4.5 |
| CART | 一个节点只能分出两个分支 | CART |

多叉树的优点是直观，树可能更浅。比如 `天气=晴/阴/雨`，一层就能展开完。

多叉树的缺点是容易分支爆炸。如果某个特征有很多取值，例如城市、用户编号、商品 ID，树会一下子长出很多分支，容易过拟合，也不容易解释。

CART 的优点是结构统一，永远是二选一问题：

```text
是否属于某个集合？
是否小于等于某个阈值？
```

这种结构很适合工程实现、剪枝和处理连续特征。

CART 的缺点是树可能更深。多叉树一层能分完的东西，CART 可能需要连续问好几次。

## 11. 连续特征怎么处理？

CART 很擅长处理连续特征。比如有一个特征叫 `年龄`，它会尝试不同阈值：

```text
年龄 <= 25 ?
年龄 <= 30 ?
年龄 <= 35 ?
```

每个阈值都会形成一个二叉划分：

```text
年龄 <= 30
├── 是
└── 否
```

然后 CART 计算每个阈值对应的加权 Gini，选择最小的那个。

## 12. 连续特征小例子：按年龄划分

| 编号 | 年龄 | 是否批准贷款 |
| ---: | ---: | --- |
| 1 | 22 | 拒绝 |
| 2 | 25 | 拒绝 |
| 3 | 28 | 批准 |
| 4 | 35 | 批准 |
| 5 | 40 | 批准 |

候选阈值通常取相邻年龄的中点：

- 23.5，来自 22 和 25
- 26.5，来自 25 和 28
- 31.5，来自 28 和 35
- 37.5，来自 35 和 40

比如阈值 26.5：

| 分支 | 样本 | 标签 |
| --- | --- | --- |
| 年龄 <= 26.5 | 22, 25 | 拒绝, 拒绝 |
| 年龄 > 26.5 | 28, 35, 40 | 批准, 批准, 批准 |

两个分支都纯，Gini 都是 0，所以加权 Gini 也是 0。

这说明 `年龄 <= 26.5` 是一个非常好的划分。

In [ ]:
age_data = [
    {'年龄': 22, '是否批准贷款': '拒绝'},
    {'年龄': 25, '是否批准贷款': '拒绝'},
    {'年龄': 28, '是否批准贷款': '批准'},
    {'年龄': 35, '是否批准贷款': '批准'},
    {'年龄': 40, '是否批准贷款': '批准'},
]

def gini_index_for_threshold(rows, feature, threshold, target):
    left = [row for row in rows if row[feature] <= threshold]
    right = [row for row in rows if row[feature] > threshold]
    total = len(rows)
    return (
        (len(left) / total) * gini([row[target] for row in left])
        + (len(right) / total) * gini([row[target] for row in right])
    )

ages = sorted(row['年龄'] for row in age_data)
thresholds = [(ages[i] + ages[i + 1]) / 2 for i in range(len(ages) - 1)]

for threshold in thresholds:
    score = gini_index_for_threshold(age_data, '年龄', threshold, target)
    print(f"年龄 <= {threshold:.1f}: 加权 Gini = {score:.4f}")


## 13. 本课小结

- CART 可以做分类树，也可以做回归树。
- CART 分类树常用 Gini 衡量节点不纯度。
- 节点越纯，Gini 越小。
- CART 每次选择加权 Gini 最小的二叉划分。
- 比较划分时，看的是整次划分后的加权 Gini，不是某一个分支的 Gini。
- 多叉树一个节点可以分出多个分支，直观但容易分支爆炸。
- CART 一个节点只分出两个分支，结构统一，但树可能更深。
- 对离散特征，CART 会寻找二叉分组。
- 对连续特征，CART 会寻找最优阈值。

记忆方式：**ID3/C4.5 喜欢问“信息减少多少”，CART 喜欢问“切完以后够不够纯”。**